In [1]:
import sys
import os
import argparse
import datasets
import tqdm
from huggingface_hub.hf_api import HfFolder
from transformers import AutoModelForCausalLM, AutoTokenizer, StoppingCriteria, StoppingCriteriaList
from peft import PeftModel
import torch
import json

print(1/0)

ZeroDivisionError: division by zero

In [ ]:
sft_model_name1 = '/mnt/localssd/models/Safe_RLHF/sft_llama_8b_final_True_chosen'
sft_model_name2 = '/mnt/localssd/models/Safe_RLHF/sft_llama_8b_final_False_chosen'
#tokenizer_name = config.params['sft_tokenizer_name']

#tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, cache_dir=cache_dir)
sft_model1 = AutoModelForCausalLM.from_pretrained(sft_model_name1,
                                                load_in_8bit=False,
                                                load_in_4bit=False,
                                                device_map="auto")

sft_model2 = AutoModelForCausalLM.from_pretrained(sft_model_name2,
                                                load_in_8bit=False,
                                                load_in_4bit=False,
                                                device_map="auto")

In [ ]:
state_dict1 = sft_model1.state_dict()
state_dict2 = sft_model2.state_dict()

# For each parameter in the state dictionaries
for param1, param2 in zip(state_dict1.items(), state_dict2.items()):
    # Unpack the parameter names and tensors
    name1, tensor1 = param1
    name2, tensor2 = param2

    # Check the names are the same (they should be, if the models are of the same architecture)
    if name1 == name2:
        try:
            # Calculate the difference between the two tensors
            difference = torch.nn.functional.mse_loss(tensor1, tensor2)
            
            print(f"Difference in {name1}: {difference}")
        except:
            print(f"Cannot compare {name1} and {name2}")
    else:
        print(f"Parameter names do not match: {name1}, {name2}")